In [ ]:
# Test 
from TCT import name_resolver
from TCT import node_normalizer
from TCT import translator_query
from TCT import translator_metakg
from TCT import visualization
from TCT import TCT

In [ ]:
from TCT.translator_resources import TranslatorResources
resources = TranslatorResources.load()

In [4]:
input_node_1 = "fanconi anemia"
input_node_info = name_resolver.lookup(input_node_1, return_top_response=False,  biolink_type="biolink:Disease")
for k in input_node_info:
    print(k)


TranslatorNode(curie='MONDO:0019391', label='Fanconi anemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[])
TranslatorNode(curie='MONDO:0009215', label='Fanconi anemia complementation group A', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[])
TranslatorNode(curie='MONDO:0011584', label='Fanconi anemia complementation group D1', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[])
TranslatorNode(curie='MONDO:0009213', label='Fanconi anemia complementation group C', types=['biolink:

In [5]:
input_node_1_id = "MONDO:0019391"
input_node_1_category = ["biolink:Disease",'biolink:DiseaseOrPhenotypicFeature']

input_node_2_category = ["biolink:Gene", 'biolink:Protein', 'biolink:Drug']

In [ ]:
from TCT import TCT
sele_predicates, sele_APIs, API_URLs = TCT.sele_predicates_API(input_node1_category=input_node_1_category,
                                                                input_node2_category=input_node_2_category,
                                                                metaKG=resources.meta_kg,
                                                                APInames=resources.api_names)

print(sele_predicates)

In [ ]:
query_json = TCT.format_query_json(subject_ids=[input_node_1_id],
                                   object_ids=[],
                                   subject_categories=input_node_1_category,
                                   object_categories=input_node_2_category,
                                   predicates=sele_predicates)

In [ ]:
result = translator_query.parallel_api_query(query_json=query_json,
                             select_APIs= sele_APIs,
                             resources=resources,
                             max_workers=len(sele_APIs))

In [ ]:
dic_graph = visualization.visualize_neighborhood_graph(result, show_label=True, height="500", width="100%")

In [10]:
dic_graph

{'gene_associated_with_condit': <networkx.classes.digraph.DiGraph at 0x122af0350>,
 '_clinical_trials_for': <networkx.classes.digraph.DiGraph at 0x122af1ad0>,
 'contributes_t': <networkx.classes.digraph.DiGraph at 0x122af2c90>,
 'treats_or_applied_or_studied_to_treat': <networkx.classes.digraph.DiGraph at 0x122afc590>,
 'genetically_associated_with': <networkx.classes.digraph.DiGraph at 0x122b06b90>,
 'contraindicated_': <networkx.classes.digraph.DiGraph at 0x122b10210>,
 'close_match': <networkx.classes.digraph.DiGraph at 0x122b10150>,
 'causes': <networkx.classes.digraph.DiGraph at 0x122b106d0>,
 'related_t': <networkx.classes.digraph.DiGraph at 0x122b10b90>,
 'marker_for': <networkx.classes.digraph.DiGraph at 0x122b11110>,
 'affects': <networkx.classes.digraph.DiGraph at 0x122b73bd0>,
 'subclass_of': <networkx.classes.digraph.DiGraph at 0x122b86e50>}

In [11]:
import networkx as nx
from pyvis.network import Network
G1 = dic_graph['marker_for']
G2 = dic_graph['gene_associated_with_condit']
G_merged = nx.compose(G1, G2)
selected_nodes = [n for n, d in G_merged.degree() if d > 1]
subgraph = G_merged.subgraph(selected_nodes).copy()

net = Network(height="1000px", width="100%", notebook=True, cdn_resources="in_line")
net.from_nx(subgraph)

# Remove edge labels before passing to PyVis
for u, v, d in subgraph.edges(data=True):
    d.pop("label", None)  # remove 'label' if it exists


for e in net.edges:
    e["title"] = "\n".join([f"{k}: {v}" for k,v in subgraph[e["from"]][e["to"]].items()])
# add title in the figure
title_html = f"<h3>merged_graph</h3>"
net.title = title_html + f"<p>Nodes: {net.num_nodes()} Edges: {net.num_edges()}</p>"
net.show("merged_graph.html")

merged_graph.html
